# 08 — Traditional ML Baselines on Full Data (LogReg + XGBoost)

**Goal**: Train LogReg and XGBoost on the **full 1.8M dataset** with TF-IDF features. Save test predictions so they can be stacked with the deep learning models in NB09 to create a hybrid ML+DL ensemble.

**Why this matters**: The deep learning models (RoBERTa-D: 0.750, ModernBERT: 0.736) struggle on hard classes like Debt Mgmt (23% correct). Traditional ML with TF-IDF captures explicit keyword patterns ("debt consolidation", "credit counseling") that transformers may miss. By stacking ML + DL predictions, the meta-classifier can learn: "trust LogReg for Debt Mgmt, trust RoBERTa for Credit Report."

**NB03b trained LogReg on a subsample** (macro-F1 = 0.754). Now we train on the full 1.8M and add XGBoost.

**Time estimate**: ~15–30 minutes total. TF-IDF vectorisation is the bottleneck, not model training.

**Hardware**: 64 GB RAM handles 1.8M × 30K sparse TF-IDF matrix easily.

In [1]:
import os, sys, time, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

SEED = 42
np.random.seed(SEED)

ROOT       = Path("..").resolve()
PROCESSED  = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "models" / "ml_baselines"
FIG_DIR    = ROOT / "reports" / "figures" / "nb08"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root:   {ROOT}")
print(f"Output: {OUTPUT_DIR}")

Root:   C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\nlp_project
Output: C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\nlp_project\models\ml_baselines


## 1 · Load Data

In [2]:
train_df = pd.read_parquet(PROCESSED / "train.parquet")
val_df   = pd.read_parquet(PROCESSED / "val.parquet")
test_df  = pd.read_parquet(PROCESSED / "test.parquet")

with open(PROCESSED / "label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)

product_names = list(label_encoders["product_encoder"].classes_)
n_classes = len(product_names)

SHORT = {
    "Credit reporting or other personal consumer reports": "Credit Report",
    "Debt collection": "Debt Collect",
    "Credit card": "Credit Card",
    "Checking or savings account": "Bank Acct",
    "Mortgage": "Mortgage",
    "Money transfer, virtual currency, or money service": "Money Xfer",
    "Student loan": "Student Loan",
    "Vehicle loan or lease": "Vehicle Loan",
    "Payday loan, title loan, personal loan, or advance loan": "Payday/Pers",
    "Debt or credit management": "Debt Mgmt",
}
short_labels = [SHORT.get(n, n) for n in product_names]

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}  Classes: {n_classes}")
print()
for i, sl in enumerate(short_labels):
    n = (train_df["product_id"] == i).sum()
    print(f"  {sl:15s}  {n:>9,}")

Train: 1,813,849  Val: 331,178  Test: 274,065  Classes: 10

  Bank Acct          129,869
  Credit Card        168,930
  Credit Report      956,601
  Debt Collect       266,081
  Debt Mgmt            1,846
  Money Xfer          48,354
  Mortgage           124,631
  Payday/Pers         34,581
  Student Loan        47,768
  Vehicle Loan        35,188


## 2 · TF-IDF Vectorisation

This is the slow step (~5–10 min on 1.8M texts). The sparse matrix fits comfortably in 64 GB RAM.

In [3]:
print("Building TF-IDF features on full training set...")
print("This may take 5-10 minutes on 1.8M texts.\n")

t0 = time.time()

tfidf = TfidfVectorizer(
    max_features=50000,         # more features than NB03b (30K) — full data can support it
    ngram_range=(1, 2),         # unigrams + bigrams
    min_df=5,                   # must appear in at least 5 docs
    max_df=0.8,                 # ignore terms in >80% of docs
    sublinear_tf=True,          # log(1 + tf) — dampens frequent terms
    stop_words="english",
    dtype=np.float32,           # float32 saves RAM vs float64
)

X_train = tfidf.fit_transform(train_df["narrative"].fillna(""))
y_train = train_df["product_id"].values

X_val = tfidf.transform(val_df["narrative"].fillna(""))
y_val = val_df["product_id"].values

X_test = tfidf.transform(test_df["narrative"].fillna(""))
y_test = test_df["product_id"].values

tfidf_time = time.time() - t0

print(f"TF-IDF complete in {tfidf_time/60:.1f} min")
print(f"  Vocabulary size: {len(tfidf.vocabulary_):,}")
print(f"  X_train: {X_train.shape}  ({X_train.nnz / X_train.shape[0]:.0f} avg non-zero per doc)")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  RAM: ~{(X_train.data.nbytes + X_val.data.nbytes + X_test.data.nbytes) / 1e9:.1f} GB (sparse)")

Building TF-IDF features on full training set...
This may take 5-10 minutes on 1.8M texts.

TF-IDF complete in 6.6 min
  Vocabulary size: 50,000
  X_train: (1813849, 50000)  (97 avg non-zero per doc)
  X_val:   (331178, 50000)
  X_test:  (274065, 50000)
  RAM: ~1.0 GB (sparse)


## 3 · Logistic Regression (Full Data)

In [4]:
print("Training Logistic Regression on full 1.8M...\n")

t0 = time.time()

lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
    n_jobs=-1,
    class_weight="balanced",
    random_state=SEED,
)
lr_model.fit(X_train, y_train)
lr_train_time = time.time() - t0

print(f"LogReg trained in {lr_train_time:.1f}s")

# ── Predictions ──
lr_val_preds = lr_model.predict(X_val)
lr_val_probs = lr_model.predict_proba(X_val)

lr_test_preds = lr_model.predict(X_test)
lr_test_probs = lr_model.predict_proba(X_test)

lr_val_macro  = f1_score(y_val, lr_val_preds, average="macro")
lr_test_macro = f1_score(y_test, lr_test_preds, average="macro")
lr_test_acc   = (y_test == lr_test_preds).mean()
lr_per_class  = f1_score(y_test, lr_test_preds, average=None)

print(f"\n  Val  Macro-F1: {lr_val_macro:.4f}")
print(f"  Test Macro-F1: {lr_test_macro:.4f}")
print(f"  Test Accuracy: {lr_test_acc:.4f}")
print()
print(classification_report(y_test, lr_test_preds, target_names=short_labels, digits=4))

Training Logistic Regression on full 1.8M...

LogReg trained in 700.5s

  Val  Macro-F1: 0.6810
  Test Macro-F1: 0.6891
  Test Accuracy: 0.8209

               precision    recall  f1-score   support

    Bank Acct     0.7983    0.7714    0.7847     23730
  Credit Card     0.7583    0.7391    0.7486     24237
Credit Report     0.9241    0.8757    0.8992    150438
 Debt Collect     0.7332    0.7236    0.7284     40040
    Debt Mgmt     0.1732    0.2638    0.2091      1232
   Money Xfer     0.6497    0.7734    0.7062     11277
     Mortgage     0.8584    0.9017    0.8795      7938
  Payday/Pers     0.4347    0.6838    0.5315      4516
 Student Loan     0.6474    0.8670    0.7413      3954
 Vehicle Loan     0.5678    0.7965    0.6630      6703

     accuracy                         0.8209    274065
    macro avg     0.6545    0.7396    0.6891    274065
 weighted avg     0.8333    0.8209    0.8253    274065

